In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


In [2]:
# Menentukan lokasi proyek dan load dataset


# Tentukan lokasi proyek berdasarkan folder kerja
current_dir = Path.cwd()

if (current_dir / "data").exists():
    project_dir = current_dir
elif (current_dir.parent / "data").exists():
    project_dir = current_dir.parent
else:
    raise FileNotFoundError(
        "Folder data tidak ditemukan. "
        "Pastikan notebook dijalankan dari folder proyek atau notebooks."
    )

# Lokasi dataset hasil feature engineering
data_path = (
    project_dir
    / "data"
    / "processed"
    / "employee_workload_feature_engineered.csv"
)

print("Lokasi proyek:", project_dir)
print("Lokasi dataset:", data_path)

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {data_path}"
    )

df = pd.read_csv(data_path)

print("Dataset berhasil dimuat.")
print("Ukuran dataset:", df.shape)

display(df.head())

Lokasi proyek: c:\MLProjects\smart-personnel-mlops
Lokasi dataset: c:\MLProjects\smart-personnel-mlops\data\processed\employee_workload_feature_engineered.csv
Dataset berhasil dimuat.
Ukuran dataset: (2000, 33)


,employee_id,department,division,position_level,employment_type,years_experience,age,working_days,working_hours,overtime_hours,...,customer_complaints,system_usage_hours,automation_level,remote_work_days,task_complexity,productivity_score,total_tasks_completed,document_completion_rate,pending_document_ratio,experience_category
0,EMP00001,Umum,Logistik,Supervisor,Tetap,2.3,31,20,158.6,55.0,...,1.0,94.5,0.252,0,0.839,0.663,234,1.0,0.0,Menengah
1,EMP00002,Pelayanan,Front Office,Senior Staff,Kontrak,6.0,24,23,193.5,1.5,...,3.0,99.1,0.133,4,0.782,0.557,286,1.0,0.0,Berpengalaman
2,EMP00003,Umum,Logistik,Staff,Kontrak,1.2,29,20,162.8,4.5,...,3.0,94.3,0.434,3,0.502,0.691,250,1.0,0.0,Pemula
3,EMP00004,Teknologi Informasi,Infrastruktur,Senior Staff,Tetap,6.4,25,21,158.2,1.7,...,1.0,116.3,0.543,6,0.417,0.796,218,1.0,0.0,Berpengalaman
4,EMP00005,Kepegawaian,Kesejahteraan,Junior Staff,Tetap,1.7,27,20,163.3,12.6,...,0.0,111.2,0.453,3,0.382,0.689,264,1.0,0.0,Pemula


In [3]:
# Pastikan kolom target tersedia
target_column = "total_tasks_completed"

if target_column not in df.columns:
    raise ValueError(
        f"Kolom target {target_column} tidak ditemukan."
    )

# Pisahkan fitur dan target
X = df.drop(
    columns=[
        "total_tasks_completed",
        "employee_id"
    ],
    errors="ignore"
)

y = df[target_column]

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)

display(X.head())

Ukuran X: (2000, 31)
Ukuran y: (2000,)


,department,division,position_level,employment_type,years_experience,age,working_days,working_hours,overtime_hours,incoming_documents,...,error_count,customer_complaints,system_usage_hours,automation_level,remote_work_days,task_complexity,productivity_score,document_completion_rate,pending_document_ratio,experience_category
0,Umum,Logistik,Supervisor,Tetap,2.3,31,20,158.6,55.0,106,...,7,1.0,94.5,0.252,0,0.839,0.663,1.0,0.0,Menengah
1,Pelayanan,Front Office,Senior Staff,Kontrak,6.0,24,23,193.5,1.5,162,...,9,3.0,99.1,0.133,4,0.782,0.557,1.0,0.0,Berpengalaman
2,Umum,Logistik,Staff,Kontrak,1.2,29,20,162.8,4.5,97,...,3,3.0,94.3,0.434,3,0.502,0.691,1.0,0.0,Pemula
3,Teknologi Informasi,Infrastruktur,Senior Staff,Tetap,6.4,25,21,158.2,1.7,122,...,0,1.0,116.3,0.543,6,0.417,0.796,1.0,0.0,Berpengalaman
4,Kepegawaian,Kesejahteraan,Junior Staff,Tetap,1.7,27,20,163.3,12.6,130,...,4,0.0,111.2,0.453,3,0.382,0.689,1.0,0.0,Pemula


In [4]:
# Pastikan kolom target tersedia
target_column = "total_tasks_completed"

if target_column not in df.columns:
    raise ValueError(
        f"Kolom target {target_column} tidak ditemukan."
    )

# Pisahkan fitur dan target
X = df.drop(
    columns=[
        "total_tasks_completed",
        "employee_id"
    ],
    errors="ignore"
)

y = df[target_column]

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)

display(X.head())

Ukuran X: (2000, 31)
Ukuran y: (2000,)


,department,division,position_level,employment_type,years_experience,age,working_days,working_hours,overtime_hours,incoming_documents,...,error_count,customer_complaints,system_usage_hours,automation_level,remote_work_days,task_complexity,productivity_score,document_completion_rate,pending_document_ratio,experience_category
0,Umum,Logistik,Supervisor,Tetap,2.3,31,20,158.6,55.0,106,...,7,1.0,94.5,0.252,0,0.839,0.663,1.0,0.0,Menengah
1,Pelayanan,Front Office,Senior Staff,Kontrak,6.0,24,23,193.5,1.5,162,...,9,3.0,99.1,0.133,4,0.782,0.557,1.0,0.0,Berpengalaman
2,Umum,Logistik,Staff,Kontrak,1.2,29,20,162.8,4.5,97,...,3,3.0,94.3,0.434,3,0.502,0.691,1.0,0.0,Pemula
3,Teknologi Informasi,Infrastruktur,Senior Staff,Tetap,6.4,25,21,158.2,1.7,122,...,0,1.0,116.3,0.543,6,0.417,0.796,1.0,0.0,Berpengalaman
4,Kepegawaian,Kesejahteraan,Junior Staff,Tetap,1.7,27,20,163.3,12.6,130,...,4,0.0,111.2,0.453,3,0.382,0.689,1.0,0.0,Pemula


In [5]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 31)
X_test: (400, 31)
y_train: (1600,)
y_test: (400,)


In [6]:
# Identifikasi tipe fitur
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Jumlah fitur numerik:", len(numeric_features))
print("Jumlah fitur kategorikal:", len(categorical_features))

# Pipeline untuk fitur numerik
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline untuk fitur kategorikal
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Menggabungkan preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline berhasil dibuat.")

Jumlah fitur numerik: 26
Jumlah fitur kategorikal: 5
Preprocessing pipeline berhasil dibuat.


C:\Users\HENDRO\AppData\Local\Temp\ipykernel_6508\3686001759.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [7]:
mlflow_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

print("Model pipeline berhasil dibuat.")

Model pipeline berhasil dibuat.


In [8]:
# Menentukan lokasi database MLflow
database_path = project_dir / "mlflow.db"

# Menggunakan SQLite sebagai tracking backend
tracking_uri = ("http://127.0.0.1:5000")

mlflow.set_tracking_uri(tracking_uri)

# Membuat atau memilih eksperimen
mlflow.set_experiment(
    "smart-personnel-workload"
)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Database:", database_path)
print("Eksperimen MLflow berhasil disiapkan.")

2026/10/04 19:49:27 INFO mlflow.tracking.fluent: Experiment with name 'smart-personnel-workload' does not exist. Creating a new experiment.


Tracking URI: http://127.0.0.1:5000
Database: c:\MLProjects\smart-personnel-mlops\mlflow.db
Eksperimen MLflow berhasil disiapkan.


In [9]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.pipeline import Pipeline



with mlflow.start_run(
    run_name="linear_regression_v1"
):

    # Training
    mlflow_pipeline.fit(X_train, y_train)

    # Prediksi
    y_pred = mlflow_pipeline.predict(X_test)

    # Evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Catat parameter
    mlflow.log_params({
        "model": "LinearRegression",
        "test_size": 0.2,
        "random_state": 42
    })

    # Catat metrik
    mlflow.log_metrics({
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    # Simpan model sebagai artefak
    mlflow.sklearn.log_model(
        sk_model=mlflow_pipeline,
        artifact_path="model",
        serialization_format="pickle"
)

    print("Training selesai.")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

2026/10/04 19:49:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/04 19:49:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training selesai.
MAE: 8.991110055635595
RMSE: 11.347714055084783
R2: 0.8884238499459844
🏃 View run linear_regression_v1 at: http://127.0.0.1:5000/#/experiments/1/runs/de7b4247135b47ca94fb33b06da13fdb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [10]:
experiment = mlflow.get_experiment_by_name(
    "smart-personnel-workload"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

display(
    runs[
        [
            "tags.mlflow.runName",
            "metrics.MAE",
            "metrics.RMSE",
            "metrics.R2",
            "status"
        ]
    ].head()
)

,tags.mlflow.runName,metrics.MAE,metrics.RMSE,metrics.R2,status
0,linear_regression_v1,8.99111,11.347714,0.888424,FINISHED


In [11]:
import mlflow
from pathlib import Path

print("Folder kerja:", Path.cwd())
print("Tracking URI:", mlflow.get_tracking_uri())

experiments = mlflow.search_experiments()

print("\nDaftar eksperimen:")
for exp in experiments:
    print("Nama:", exp.name)
    print("ID:", exp.experiment_id)
    print("Lokasi artefak:", exp.artifact_location)

Folder kerja: c:\MLProjects\smart-personnel-mlops\notebooks
Tracking URI: http://127.0.0.1:5000

Daftar eksperimen:
Nama: smart-personnel-workload
ID: 1
Lokasi artefak: mlflow-artifacts:/1
Nama: Default
ID: 0
Lokasi artefak: mlflow-artifacts:/0


In [12]:
import mlflow

mlflow.set_experiment("smart-personnel-workload")

with mlflow.start_run(run_name="test-mlflow"):
    mlflow.log_param("test", "koneksi-berhasil")
    mlflow.log_metric("test_score", 0.95)

print("Eksperimen:", mlflow.get_experiment_by_name(
    "smart-personnel-workload"
).name)

print("Run uji berhasil dicatat.")

🏃 View run test-mlflow at: http://127.0.0.1:5000/#/experiments/1/runs/5aec8c8f95f94ceb9f17fe3c35b4d88d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Eksperimen: smart-personnel-workload
Run uji berhasil dicatat.


In [13]:
import mlflow

with mlflow.start_run(run_name="test-http-connection"):
    mlflow.log_param("metode", "HTTP")
    mlflow.log_metric("koneksi", 1.0)

print("Run berhasil dikirim ke MLflow server.")

runs = mlflow.search_runs(
    experiment_names=["smart-personnel-workload"]
)

print("Jumlah run:", len(runs))
display(runs[["run_id", "status", "metrics.koneksi"]])

🏃 View run test-http-connection at: http://127.0.0.1:5000/#/experiments/1/runs/3516f1351a9645f3804e31c2af4c80d7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run berhasil dikirim ke MLflow server.
Jumlah run: 3


,run_id,status,metrics.koneksi
0,3516f1351a9645f3804e31c2af4c80d7,FINISHED,1.0
1,5aec8c8f95f94ceb9f17fe3c35b4d88d,FINISHED,NaN
2,de7b4247135b47ca94fb33b06da13fdb,FINISHED,NaN
